# Day 3: Series and DataFrames Deep Dive

## Learning Objectives
By the end of this session, you will be able to:
- Understand the difference between Series and DataFrames
- Create and manipulate Series
- Use `loc` for label-based selection
- Use `iloc` for position-based selection
- Apply boolean indexing to filter data
- Set and modify values in DataFrames
- Work with DataFrame indices

**Duration**: 45 minutes

**Prerequisites**: Notebook 10 (Introduction to Pandas)

---

## 1. Review: What You Know So Far

Yesterday you learned:
- Creating DataFrames from dictionaries
- Loading CSV files with `pd.read_csv()`
- Exploring data with `head()`, `info()`, `describe()`
- Selecting columns with `df["column"]`

Today we'll go deeper into **how** DataFrames work!

In [ ]:
# Setup: Create sample data for this lesson
# This DataFrame will be used throughout to demonstrate Series and DataFrame concepts

import pandas as pd

# Create a DataFrame with multiple data types:
# - String columns: ProductName, Category
# - Numeric columns: Price, Units
# - Boolean column: InStock
products = pd.DataFrame({
    "ProductName": ["Suite A", "Suite B", "Suite C", "Analytics", "Enterprise"],
    "Category": ["Premium", "Standard", "Premium", "Premium", "Enterprise"],
    "Price": [199, 99, 299, 399, 799],
    "Units": [150, 250, 80, 120, 45],
    "InStock": [True, True, True, False, True]
})

print("Our sample data:")
print(products)

---
## 2. Understanding Series

A **Series** is a single column of data with an index. Think of it as a labeled list.

### Demo 2.1: What is a Series?

In [ ]:
# Demo: What is a Series?
# A Series is a ONE-DIMENSIONAL labeled array (like a single column)
# When you select ONE column from a DataFrame, you get a Series

# Select the Price column using bracket notation
# Syntax: df["column_name"] returns a Series
prices = products["Price"]

print("The Price column:")
print(prices)

# Verify the type - it's pandas.core.series.Series, NOT DataFrame
print(f"\nType: {type(prices)}")

In [ ]:
# Demo: Components of a Series
# A Series has THREE main components:

# 1. VALUES - The actual data stored in the Series
# Access with .values attribute - returns a numpy array
print("Values:")
print(prices.values)

# 2. INDEX - The labels for each value (like row numbers)
# Access with .index attribute
# Default index is 0, 1, 2, 3... (RangeIndex)
print("\nIndex:")
print(prices.index)

# 3. NAME - The name of the Series (usually the column name)
# Access with .name attribute
print(f"\nName: {prices.name}")

### Demo 2.2: Creating Series Directly

In [ ]:
# Demo: Create a Series from a List
# You can create Series directly, not just by extracting from DataFrames
# Syntax: pd.Series(data)

# With automatic numeric index (0, 1, 2...)
revenue_list = pd.Series([125000, 98000, 67000])
print("From list (auto index):")
print(revenue_list)

In [ ]:
# Demo: Create Series with Custom Index
# Custom indices make data more meaningful and easier to access
# Syntax: pd.Series(data, index=labels, name="series_name")

revenue = pd.Series(
    [125000, 98000, 67000],           # The data values
    index=["EMEA", "AMER", "APAC"],   # Custom labels for each value
    name="Q1_Revenue"                  # Name for the Series
)

print("With custom index:")
print(revenue)

In [ ]:
# Demo: Access Series Values by Index Label
# With custom index, use the label names to access values
# Syntax: series["label"] for single value
# Syntax: series[["label1", "label2"]] for multiple values

# Access single value by label
print(f"EMEA Revenue: €{revenue['EMEA']:,}")
print(f"APAC Revenue: €{revenue['APAC']:,}")

# Access multiple values - pass a LIST of labels
# Note the double brackets: outer for selection, inner for list
print("\nEMEA and APAC:")
print(revenue[["EMEA", "APAC"]])

### Demo 2.3: Series Operations

In [ ]:
# Demo: Series Aggregation Methods
# Series has built-in methods for common calculations
# These return a SINGLE value summarizing the data

# Basic statistics
print(f"Sum: €{revenue.sum():,}")      # Total of all values
print(f"Mean: €{revenue.mean():,.2f}") # Average
print(f"Max: €{revenue.max():,}")      # Highest value
print(f"Min: €{revenue.min():,}")      # Lowest value

# idxmax() returns the INDEX LABEL of the maximum value
# Very useful for finding "which region/product/etc had the highest..."
print(f"\nBest region: {revenue.idxmax()}")

In [ ]:
# Demo: Vectorized Operations on Series
# Operations apply to ALL elements automatically (no loops needed!)
# This is called "vectorization" - much faster than Python loops

print("Original revenue:")
print(revenue)

# Multiply all values by 1.10 (10% growth)
# Each value gets multiplied individually
print("\nWith 10% growth:")
print(revenue * 1.10)

# Comparison operators also work element-wise
# Returns a Series of True/False (Boolean Series)
print("\nBoolean comparison (>100000):")
print(revenue > 100000)

### Exercise 1: Working with Series

Create a Series of monthly sales and perform operations:

In [ ]:
# Exercise 1: Working with Series
# Task: Create a Series and perform aggregation operations

# Create a Series with months as index
monthly_sales = pd.Series(
    [15000, 18000, 16500, 19000, 17500, 21000],
    index=["Jan", "Feb", "Mar", "Apr", "May", "Jun"],
    name="Sales"
)

print("Monthly Sales:")
print(monthly_sales)

# 1. What is the total sales?
# Use .sum() method to add all values
total = monthly_sales.___
print(f"\nTotal: €{total:,}")

# 2. What is the average monthly sales?
# Use .mean() method to calculate average
avg = monthly_sales.___
print(f"Average: €{avg:,.2f}")

# 3. Which month had the highest sales?
# Use .idxmax() to get the INDEX (month name) of the max value
best_month = monthly_sales.___
print(f"Best month: {best_month}")

# 4. Access sales for March
# Use bracket notation with the index label
march_sales = monthly_sales[___]  # Use "Mar" as the label
print(f"March sales: €{march_sales:,}")

---
## 3. DataFrames = Collection of Series

A **DataFrame** is essentially multiple Series that share the same index.

In [ ]:
# Demo: DataFrame = Collection of Series
# A DataFrame is essentially multiple Series that share the same index
# Think of it as Series stacked side by side (columns)

print(products)
print(f"\nType: {type(products)}")

In [ ]:
# Demo: Each Column is a Series
# When you select individual columns, you get Series objects
# All columns share the same index (row labels)

# Check the type of individual columns
print("Price column type:", type(products["Price"]))
print("Units column type:", type(products["Units"]))

# All columns share the DataFrame's index
print("\nDataFrame index:")
print(products.index)

---
## 4. Selection with `loc` (Label-Based)

**`loc`** selects data by **labels** (index and column names).

Syntax: `df.loc[row_labels, column_labels]`

### Demo 4.1: Basic loc Usage

In [ ]:
# Our data (with default numeric index)
print(products)
print("\nIndex labels are: 0, 1, 2, 3, 4")

In [ ]:
# Demo: loc - Select Single Row by Index Label
# loc uses the INDEX LABELS (not positions)
# Syntax: df.loc[row_label]

# Select the row with index label 0
# Returns a Series where column names become the index
print("Row with index label 0:")
print(products.loc[0])

In [ ]:
# Demo: loc - Select Multiple Rows
# Pass a LIST of index labels to get multiple rows
# Syntax: df.loc[[label1, label2, label3]]

print("Rows 0, 2, and 4:")
print(products.loc[[0, 2, 4]])

In [ ]:
# Demo: loc - Select Specific Row AND Column
# Syntax: df.loc[row_label, column_name]
# Returns a single value (scalar)

print("Row 0, Price column:")
print(products.loc[0, "Price"])

In [ ]:
# Demo: loc - Select Multiple Rows AND Multiple Columns
# Syntax: df.loc[[row_labels], [column_names]]
# Returns a DataFrame

print("Rows 0-2, ProductName and Price:")
print(products.loc[[0, 1, 2], ["ProductName", "Price"]])

### Demo 4.2: loc with Slicing

**Important:** With `loc`, slicing is **inclusive** of the endpoint!

In [ ]:
# Demo: loc with Slicing - INCLUSIVE Endpoint!
# IMPORTANT: Unlike Python lists, loc slicing INCLUDES the endpoint
# Syntax: df.loc[start:end] includes both start AND end

# Slice rows 1 to 3 - this INCLUDES row 3!
# (In normal Python, 1:3 would give indices 1 and 2 only)
print("Rows 1 to 3 (inclusive):")
print(products.loc[1:3])

In [ ]:
# Demo: loc - Slice Both Rows AND Columns
# You can slice column names too (alphabetically)
# Syntax: df.loc[row_start:row_end, col_start:col_end]

print("Rows 0-2, columns ProductName to Price:")
print(products.loc[0:2, "ProductName":"Price"])

In [ ]:
# Demo: loc - All Rows with Specific Columns
# Use : (colon alone) to select ALL rows
# Syntax: df.loc[:, [column_list]]

print("All rows, just ProductName and Category:")
print(products.loc[:, ["ProductName", "Category"]])

### Demo 4.3: loc with Custom Index

In [ ]:
# Demo: loc with Meaningful Custom Index
# set_index() converts a column into the row index
# This makes loc more intuitive - select by product name!
# Syntax: df.set_index("column_name")

products_indexed = products.set_index("ProductName")
print("With ProductName as index:")
print(products_indexed)

In [ ]:
# Demo: Select by Custom Index Label
# Now we can use product names directly!
# Syntax: df.loc["label_value"]

print("Select 'Suite A':")
print(products_indexed.loc["Suite A"])

In [ ]:
# Demo: Multiple Custom Index Selections
# Select multiple products and specific columns
# Syntax: df.loc[[labels], [columns]]

print("Suite A and Analytics, just Price and Units:")
print(products_indexed.loc[["Suite A", "Analytics"], ["Price", "Units"]])

---
## 5. Selection with `iloc` (Position-Based)

**`iloc`** selects data by **integer position** (like list indexing).

Syntax: `df.iloc[row_positions, column_positions]`

### Demo 5.1: Basic iloc Usage

In [ ]:
print(products)
print("\nPositions are: 0, 1, 2, 3, 4 (like list indices)")

In [ ]:
# Demo: iloc - Select by Position
# iloc uses INTEGER POSITIONS (like list indexing)
# Syntax: df.iloc[position]

# First row (position 0)
print("First row:")
print(products.iloc[0])

In [ ]:
# Demo: iloc - Negative Indexing
# Like Python lists, -1 means "last", -2 means "second to last", etc.
# Syntax: df.iloc[-1]

print("Last row:")
print(products.iloc[-1])

In [ ]:
# Demo: iloc - Row and Column by Position
# Syntax: df.iloc[row_position, column_position]
# Both are integers (0, 1, 2, etc.)

# First row (0), second column (1)
print("Row 0, Column 1:")
print(products.iloc[0, 1])

### Demo 5.2: iloc with Slicing

**Important:** With `iloc`, slicing is **exclusive** of the endpoint (like normal Python)!

In [ ]:
# Demo: iloc Slicing - EXCLUSIVE Endpoint (Like Normal Python!)
# IMPORTANT: Unlike loc, iloc slicing EXCLUDES the endpoint
# This is consistent with Python list slicing
# Syntax: df.iloc[start:end] includes start but NOT end

# Positions 0, 1, 2 (NOT 3)
print("First 3 rows (iloc[0:3]):")
print(products.iloc[0:3])

In [ ]:
# Demo: iloc - Slice Rows and Columns
# Syntax: df.iloc[row_start:row_end, col_start:col_end]

# First 2 rows (positions 0, 1), first 3 columns (positions 0, 1, 2)
print("First 2 rows, first 3 columns:")
print(products.iloc[0:2, 0:3])

In [ ]:
# Demo: iloc - Last N Rows
# Use negative indexing with slicing
# Syntax: df.iloc[-n:]

print("Last 2 rows:")
print(products.iloc[-2:])

In [ ]:
# Demo: iloc - Step/Stride
# Syntax: df.iloc[start:end:step]
# Step of 2 means "every other row"

print("Every other row (step=2):")
print(products.iloc[::2])

### Key Difference: loc vs iloc

| Feature | `loc` | `iloc` |
|---------|-------|--------|
| Based on | Labels | Positions |
| Slicing | **Inclusive** endpoint | **Exclusive** endpoint |
| Use when | You know the labels | You know the positions |
| Example | `df.loc["EMEA"]` | `df.iloc[0]` |

### Exercise 2: loc and iloc Practice

In [ ]:
# Exercise 2: loc and iloc Practice
# Task: Practice both selection methods

print(products)

# 1. Use iloc to get the first 2 rows
# iloc uses positions: [0:2] gets positions 0 and 1
first_two = products.iloc[___]  # Use 0:2 or [:2]
print("\n1. First 2 rows:")
print(first_two)

# 2. Use iloc to get the last row
# Use negative indexing: -1 is the last position
last_row = products.iloc[___]  # Use -1
print("\n2. Last row:")
print(last_row)

# 3. Use loc to get rows 1, 2, 3
# loc uses labels: pass a list [1, 2, 3]
# Note: with default index, labels ARE the numbers
middle_rows = products.loc[___]  # Use [1, 2, 3] or 1:3
print("\n3. Rows 1, 2, 3:")
print(middle_rows)

# 4. Use loc to get the Price of row 2
# Syntax: df.loc[row_label, column_name]
price_row2 = products.loc[___, ___]  # Use 2, "Price"
print(f"\n4. Price of row 2: €{price_row2}")

---
## 6. Boolean Selection (Filtering)

The most powerful selection method! Filter rows based on conditions.

### Demo 6.1: How Boolean Selection Works

In [ ]:
print(products)

In [ ]:
# Demo: How Boolean Selection Works - Step by Step
# Boolean selection is THE most powerful filtering method

# Step 1: Create a boolean condition
# This compares each value and returns True/False for each row
condition = products["Price"] > 200

print("Boolean mask (Price > 200):")
print(condition)

# Note the type - it's a Series of boolean values
print(f"\nType: {type(condition)}")

In [ ]:
# Demo: Apply Boolean Mask to Filter
# Step 2: Pass the boolean mask to df[] to filter rows
# Only rows where condition is True are kept

expensive_products = products[condition]

print("Products where Price > 200:")
print(expensive_products)

In [ ]:
# Demo: One-Line Boolean Selection
# Usually written in one line (condition inside brackets)
# Syntax: df[df["column"] comparison_operator value]

expensive = products[products["Price"] > 200]

print("Same result (one line):")
print(expensive)

### Demo 6.2: Different Conditions

In [ ]:
# Demo: Filter for Equality
# Use == for exact match (double equals!)
# Syntax: df[df["column"] == "value"]

premium = products[products["Category"] == "Premium"]
print("Premium products:")
print(premium)

In [ ]:
# Demo: Filter for Not Equal
# Use != for "not equal to"
# Syntax: df[df["column"] != "value"]

not_premium = products[products["Category"] != "Premium"]
print("Non-Premium products:")
print(not_premium)

In [ ]:
# Demo: Filter on Boolean Column
# For True/False columns, you can compare to True
# Or simply use the column directly (it's already boolean!)

# Explicit comparison
in_stock = products[products["InStock"] == True]

# Simpler: just use the column (preferred style)
in_stock = products[products["InStock"]]

print("In-stock products:")
print(in_stock)

### Demo 6.3: Combining Conditions

In [ ]:
# Demo: Combining Conditions with AND
# Use & (ampersand) for AND - BOTH conditions must be True
# CRITICAL: Wrap EACH condition in parentheses!
# Syntax: df[(condition1) & (condition2)]

premium_and_expensive = products[
    (products["Category"] == "Premium") & 
    (products["Price"] > 200)
]

print("Premium AND Price > 200:")
print(premium_and_expensive)

In [ ]:
# Demo: Combining Conditions with OR
# Use | (pipe) for OR - EITHER condition can be True
# CRITICAL: Wrap EACH condition in parentheses!
# Syntax: df[(condition1) | (condition2)]

cheap_or_high_stock = products[
    (products["Price"] < 150) | 
    (products["Units"] > 200)
]

print("Price < 150 OR Units > 200:")
print(cheap_or_high_stock)

In [ ]:
# Demo: Negating a Condition with NOT
# Use ~ (tilde) to invert True/False
# Syntax: df[~condition]

# For boolean columns, ~ gives you the opposite
not_in_stock = products[~products["InStock"]]

print("NOT in stock:")
print(not_in_stock)

### Demo 6.4: Boolean Selection with loc

In [ ]:
# Demo: Boolean Selection + Column Selection with loc
# Combine filtering with column selection for clean output
# Syntax: df.loc[boolean_condition, [column_list]]

result = products.loc[
    products["Price"] > 200,           # Rows: where Price > 200
    ["ProductName", "Price", "Units"]  # Columns: only these three
]

print("Expensive products (selected columns):")
print(result)

### Exercise 3: Boolean Selection

In [ ]:
# Exercise 3: Boolean Selection
# Task: Filter the products DataFrame using conditions

print(products)

# 1. Filter for products with Units >= 100
# Use comparison operator >=
high_stock = products[products[___] >= ___]  # "Units", 100
print("\n1. High stock (Units >= 100):")
print(high_stock)

# 2. Filter for Standard category products
# Use == for exact string match
standard = products[products[___] == ___]  # "Category", "Standard"
print("\n2. Standard products:")
print(standard)

# 3. Filter for Premium products that are in stock
# Combine two conditions with & (and parentheses!)
# Condition 1: Category == "Premium"
# Condition 2: InStock is True (just use the column)
premium_in_stock = products[
    (products["Category"] == "Premium") & 
    (products[___])  # Just "InStock" - it's already boolean
]
print("\n3. Premium AND in stock:")
print(premium_in_stock)

---
## 7. Setting and Modifying Values

You can use `loc` and `iloc` to **change** values in a DataFrame.

### Demo 7.1: Setting a Single Value

In [ ]:
# Demo: Create a Copy to Modify
# IMPORTANT: Use .copy() when you want to modify without affecting original
# This avoids the "SettingWithCopyWarning"

df = products.copy()

print("Before:")
print(df)

In [ ]:
# Demo: Set a Single Value with loc
# Syntax: df.loc[row_label, column_name] = new_value

# Change the price of row 0 to 209
df.loc[0, "Price"] = 209

print("After changing row 0 Price to 209:")
print(df)

### Demo 7.2: Setting Multiple Values

In [ ]:
# Demo: Set Entire Column (All Rows)
# Assign a single value to create a new column with that value everywhere
# Syntax: df["new_column"] = value

df["Discount"] = 0.10  # 10% discount for all products

print("Added Discount column:")
print(df)

In [ ]:
# Demo: Set Values Based on Condition
# Use loc with boolean condition to update specific rows
# Syntax: df.loc[condition, "column"] = new_value

# Set discount to 15% only for Premium products
df.loc[df["Category"] == "Premium", "Discount"] = 0.15

print("Premium products now have 15% discount:")
print(df)

In [ ]:
# Demo: Conditional Update Based on Another Column
# Update InStock to False for items with low stock

df.loc[df["Units"] < 100, "InStock"] = False

print("Low stock items marked as out of stock:")
print(df)

### Demo 7.3: Creating Calculated Columns

In [ ]:
# Demo: Calculated Columns Using Other Columns
# Create new columns from calculations on existing columns
# Operations are vectorized (apply to all rows automatically)

# DiscountedPrice = Price * (1 - Discount)
df["DiscountedPrice"] = df["Price"] * (1 - df["Discount"])

print("With DiscountedPrice:")
print(df[["ProductName", "Price", "Discount", "DiscountedPrice"]])

### Important Warning: Avoid Chained Assignment!

In [ ]:
# Demo: Avoid Chained Assignment!
# WRONG: df[condition]["col"] = value  (may not work, gives warning)
# RIGHT: df.loc[condition, "col"] = value

# BAD (chained assignment - avoid this!):
# df[df["Price"] > 200]["Status"] = "Premium"

# GOOD (use loc):
df.loc[df["Price"] > 200, "Status"] = "Expensive"
df.loc[df["Price"] <= 200, "Status"] = "Affordable"

print("With Status column:")
print(df[["ProductName", "Price", "Status"]])

---
## 8. Working with the Index

The index is powerful but can be confusing. Here's how to work with it.

### Demo 8.1: Setting the Index

In [ ]:
# Demo: Default Numeric Index
# By default, DataFrames have a RangeIndex (0, 1, 2, 3...)

df = products.copy()
print("Default numeric index:")
print(df)

In [ ]:
# Demo: Set a Column as the Index
# set_index() moves a column to become the row index
# Syntax: df.set_index("column_name")
# Note: This returns a NEW DataFrame (original unchanged unless you reassign)

df_indexed = df.set_index("ProductName")

print("With ProductName as index:")
print(df_indexed)

In [ ]:
# Demo: Select Using Custom Index
# Now loc uses the product names as labels!

print("Select 'Suite A':")
print(df_indexed.loc["Suite A"])

### Demo 8.2: Resetting the Index

In [ ]:
# Demo: Reset Index Back to Default
# reset_index() converts the index back to a column
# Syntax: df.reset_index()
# The old index becomes a regular column again

df_reset = df_indexed.reset_index()

print("After reset_index():")
print(df_reset)

### Demo 8.3: Sorting

In [ ]:
# Demo: Sort by Column Values
# sort_values() reorders rows based on column values
# Syntax: df.sort_values("column", ascending=True/False)
# ascending=True (default): smallest to largest
# ascending=False: largest to smallest

df_sorted = products.sort_values("Price", ascending=False)

print("Sorted by Price (descending):")
print(df_sorted)

In [ ]:
# Demo: Sort by Multiple Columns
# Pass a LIST of columns and a LIST of ascending values
# Sorts by first column, then by second within ties

df_multi_sorted = products.sort_values(
    ["Category", "Price"],      # Sort by Category first, then Price
    ascending=[True, False]     # Category A-Z, Price high-low
)

print("Sorted by Category (A-Z), then Price (high-low):")
print(df_multi_sorted)

---
## Challenge Exercise: Complete Data Selection

Practice all the selection methods:

In [ ]:
# Challenge Exercise: Complete Data Selection
# Task: Practice all selection methods on sales data

# Create sales data
sales = pd.DataFrame({
    "OrderID": ["ORD-001", "ORD-002", "ORD-003", "ORD-004", "ORD-005"],
    "Customer": ["Acme Corp", "TechStart", "Global Ind", "InnovateCo", "Acme Corp"],
    "Product": ["Suite A", "Suite B", "Analytics", "Suite A", "Analytics"],
    "Amount": [1500, 990, 3990, 1500, 3990],
    "Paid": [True, True, False, True, False]
})

print("Sales Data:")
print(sales)

# TASKS:
print("\n" + "="*50)

# 1. Use iloc to get the first 3 orders
# iloc uses positions: [:3] gets positions 0, 1, 2
first_three = sales.iloc[___]  # Use :3 or 0:3
print("\n1. First 3 orders:")
print(first_three)

# 2. Use loc to get OrderID and Amount for rows 1, 2, 3
# loc syntax: df.loc[row_labels, [column_list]]
subset = sales.loc[___, ___]  # Use 1:3 or [1,2,3], ["OrderID", "Amount"]
print("\n2. OrderID and Amount for rows 1-3:")
print(subset)

# 3. Filter for unpaid orders (Paid == False)
# Boolean filter: df[df["column"] == value]
# Or for False: df[~df["column"]] since Paid is boolean
unpaid = sales[___]  # sales[sales["Paid"] == False] or sales[~sales["Paid"]]
print("\n3. Unpaid orders:")
print(unpaid)

# 4. Filter for Acme Corp orders over €1000
# Combine conditions with & and parentheses
acme_large = sales[
    (sales["Customer"] == "Acme Corp") &
    (sales["Amount"] > ___)  # 1000
]
print("\n4. Acme Corp orders > €1000:")
print(acme_large)

# 5. Add a "Status" column: "Paid" if Paid is True, else "Pending"
# Use loc with boolean condition to set values
sales["Status"] = "Pending"  # Default value for all
sales.loc[sales["Paid"], "Status"] = "Paid"  # Update where Paid is True
print("\n5. With Status column:")
print(sales)

---
## Summary

**You've learned:**
- ✓ Series: single column with index
- ✓ DataFrames: multiple Series sharing an index
- ✓ `loc`: label-based selection (inclusive slicing)
- ✓ `iloc`: position-based selection (exclusive slicing)
- ✓ Boolean selection for filtering
- ✓ Combining conditions with `&`, `|`, `~`
- ✓ Setting values with `loc`
- ✓ Working with indices

**Key Takeaways:**
- Use `loc` when you know the labels (names)
- Use `iloc` when you know the positions (numbers)
- Boolean selection is the most powerful filtering method
- Always use `loc` for setting values to avoid warnings
- Remember: `loc` slices are inclusive, `iloc` slices are exclusive

**Next:** Advanced Filtering Techniques (Notebook 14)